In [1]:
import json
import sqlite3
import pandas as pd

def fetch_notegroup_json(db_path: str, table_name: str, notegroup_id: int) -> str:
    """
    Fetch record(s) for a given notegroupID from `table_name` and return
    them as a JSON string.

    Format:
    - For "notegroups": notegroupID IS the primary key, so there's only
      one record. Returned as a flat dict: {"date": ..., "data_source_category": ...}
      (NOT wrapped in a list, and NOT nested under the table name).
    - For all other tables ("participants", "questions", "answers"):
      returned as {"<table_name>": [record, record, ...]}, one entry per
      row belonging to that notegroup.
    """
    records = _fetch_records_by_notegroup(db_path, table_name, notegroup_id)

    if table_name == "notegroups":
        record = records[0] if records else {}
        record = {k: v for k, v in record.items() if k in ("date", "data_source_category")}
        json_str = json.dumps(record, ensure_ascii=False)
        return json_str

    processed = []
    for record in records:
        if table_name == "participants":
            record.pop("notegroupID", None)
            record.pop("country_staying_in", None)
            record.pop("labor_market_region", None)

            projectID_age = record.pop("projectID_age")
            age = next(iter(projectID_age.values()))

            # rebuild dict so "age" lands right after "gender"
            reordered = {}
            for key, value in record.items():
                reordered[key] = value
                if key == "gender":
                    reordered["age"] = age
            record = reordered

        elif table_name == "questions":
            record.pop("notegroupID", None)
            record.pop("question_type", None)

        elif table_name == "answers":
            record.pop("notegroupID", None)
            record.pop("projectID", None)
            record.pop("answer_extraction_LLM", None)
            record.pop("sentiment_score_LLM", None)

        processed.append(record)

    result = {table_name: processed}
    json_str = json.dumps(result, ensure_ascii=False)
    return json_str


def _fetch_records_by_notegroup(db_path: str, table: str, notegroup_id: int) -> list[dict]:
    """Fetch all rows for a given notegroupID from `table`, decoding JSONB columns."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.execute(
            f"SELECT * FROM {table} WHERE notegroupID = ?", (notegroup_id,)
        )
        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()

    records = []
    for row in rows:
        record = {}
        for col, val in zip(columns, row):
            if isinstance(val, str):
                try:
                    val = json.loads(val)  # restore JSONB columns
                except (json.JSONDecodeError, ValueError):
                    pass
            record[col] = val
        records.append(record)
    return records

def _fetch_starting_ids_for_notegroup(db_path: str, notegroup_id: int) -> dict:
    """
    Fetch the starting (minimum) ID already assigned to this notegroup's
    records in participants, questions, and answers. This is only used to
    give the refiner prompt the same ID-numbering context the original
    extraction prompt had — the refiner itself is instructed not to modify
    ID fields, so this is descriptive context, not something it acts on.
    """
    id_cols = {
        "participants": "participantID",
        "questions": "questionID",
        "answers": "answerID",
    }
    starting_ids = {}
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        for table, id_col in id_cols.items():
            cursor.execute(
                f"SELECT MIN({id_col}) FROM {table} WHERE notegroupID = ?",
                (notegroup_id,)
            )
            row = cursor.fetchone()
            starting_ids[id_col] = row[0] if row and row[0] is not None else None
    return starting_ids

def display_json_df(json_str: str, title: str = "") -> None:
    """Parse a JSON string (either a flat single-record dict, e.g. 1recordT,
    or a dict wrapping a list of records under the table name, e.g.
    {"participants": [...]}) into a DataFrame and print it."""
    data = json.loads(json_str)

    if isinstance(data, dict) and len(data) == 1 and isinstance(next(iter(data.values())), list):
        # wrapped list-of-records, e.g. {"participants": [...]}
        records = next(iter(data.values()))
    elif isinstance(data, dict):
        # flat single-record dict, e.g. 1recordT
        records = [data]
    else:
        records = data  # already a list

    df = pd.DataFrame(records)

    if title:
        print(f"\n{title}")
    print(df.to_string())

def reduce_participants(output_p: str) -> str:
    data = json.loads(output_p)

    # unwrap if nested under a key e.g. {"participants": [...]}
    if isinstance(data, dict):
        data = next(iter(data.values()))
    reduced = [
        {k: v for k, v in record.items() if v not in ("", None, [], {})}
        for record in data
    ]
    return json.dumps(reduced, ensure_ascii=False)

def reduce_questions(output_q: str) -> str:
    data = json.loads(output_q)

    if isinstance(data, dict):
        data = next(iter(data.values()))
    reduced = [
        {"questionID": record["questionID"], "question_content": record["question_content"]}
        for record in data
    ]
    return json.dumps(reduced, ensure_ascii=False)


In [2]:
from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor
from oral_notes.s2_transform.participant_llm_reducer import ParticipantReducer
from oral_notes.prompt_combiner_v3 import PromptCombiner
from openai import OpenAI
from config.config import OPENAI_API_KEY
from utils.html_viewer import show

def refine_notegroup(
    notegroup_id: int,
    task: str,
    llm_model: str = "gpt-5.1",
    max_rounds: int = 3
) -> str:
    """
    Refine the previously-extracted output for a given notegroupID and task
    ("1recordT", "participants", "questions", or "answers"): re-loads the
    source transcript(s) and file path, fetches the existing DB result(s),
    and asks the refiner LLM to either "pass" it or return a corrected
    JSON object.
    """
    # ── 1. Look up source URLs for this notegroup ───────────────────────────
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT project_name, phase, note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        row = cursor.fetchone()
        if row is None:
            raise ValueError(f"No notegroup found with ID {notegroup_id}")
        project_name, phase, note_url_qa, note_url_participant = row

    # ── 2. Re-load and extract text + file path(s), same as ETL_baseline ───
    file_loader = GoogleDriveLoader(service_account_file)
    extractor = TextExtractor()

    all_texts = {}
    all_drive_paths = {}
    for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
        if not url:
            continue
        result = file_loader.load(url)
        text = extractor.extract(result)
        all_texts[label] = f"[Data source: {result['name']}]\n{text}"
        all_drive_paths[label] = result['drive_path']
    combined_drive_paths = "|".join(all_drive_paths.values())

    # ── 3. Run participant reduction if applicable, same as ETL_baseline ───
    has_participant = "PARTICIPANT" in all_texts
    if has_participant:
        reducer = ParticipantReducer(
            pipeline_type=pipeline_type,
            prompt_path_ParReducer=prompt_path_ParReducer,
            all_texts=all_texts,
            notegroup_id=notegroup_id,
        )
        combined_text = reducer.build_combined_text()
    else:
        combined_text = all_texts["QA"]

    text_doc = combined_text
    file_path_doc = combined_drive_paths

    # ── 4. Fetch the initial DB result for this task/notegroup ─────────────
    db_table = "notegroups" if task == "1recordT" else task
    current_result = fetch_notegroup_json(DB_PATH, db_table, notegroup_id)
    display_json_df(current_result, title=f"Initial DB result — task={task}, notegroupID={notegroup_id}")

    # ── 5. Set up prompt combiner + LLM client (reused across rounds) ───────
    combiner = PromptCombiner(schema_path=schema_path)

    if task == "1recordT":
        system_prompt = combiner.load_prompts_system(prompt_path_refiner_1recordT)
    else:
        system_prompt = combiner.build_prompt_system_refiner_pqa(prompt_path_refiner_pqa, task)

    pass_placeholder = json.loads(combiner.build_pass_placeholder(task))

    # ── 5b. Task-specific context: starting IDs, and p/q reference for answers
    starting_ids = {}
    output_reduced_participants_pasttask = None
    output_reduced_questions_pasttask = None

    if task in ("participants", "questions", "answers"):
        starting_ids = _fetch_starting_ids_for_notegroup(DB_PATH, notegroup_id)

    if task == "answers":
        output_p = fetch_notegroup_json(DB_PATH, "participants", notegroup_id)
        output_q = fetch_notegroup_json(DB_PATH, "questions", notegroup_id)
        output_reduced_participants_pasttask = reduce_participants(output_p)
        output_reduced_questions_pasttask = reduce_questions(output_q)

    client = OpenAI(
        api_key=OPENAI_API_KEY,
        base_url="https://llmproxy.uva.nl/v1",
        timeout=500.0
    )

    # ── 6. Refinement loop ────────────────────────────────────────────────────
    for round_num in range(1, max_rounds + 1):
        if task == "1recordT":
            user_prompt = combiner.build_prompt_user_refiner_1recordT(
                prompt_path=prompt_path_refiner_1recordT,
                file_path_doc=file_path_doc,
                text_doc=text_doc,
                json_result_lastcall=current_result
            )
        else:
            user_prompt = combiner.build_prompt_user_refiner_pqa(
                prompt_path=prompt_path_refiner_pqa,
                task=task,
                text_doc=text_doc,
                json_result_lastcall=current_result,
                starting_ids=starting_ids,
                output_reduced_participants_pasttask=output_reduced_participants_pasttask,
                output_reduced_questions_pasttask=output_reduced_questions_pasttask,
            )

        #show(user_prompt, title=f"{notegroup_id} | {task}_refiner| prompt")

        response = client.chat.completions.create(
            model=llm_model,
            temperature=0,
            seed=42,
            response_format=combiner.to_json_schema(task),
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        result = response.choices[0].message.content.strip()
        parsed = json.loads(result)
        display_json_df(result, title=f"Round {round_num}/{max_rounds} — task={task}, notegroupID={notegroup_id}")

        if parsed == json.loads(current_result):
            print(f"No change from previous round on round {round_num}, stopping early.")
            return current_result
        if parsed == pass_placeholder:
            print(f"Passed on round {round_num}, returning previous result.")
            return current_result
        current_result = result

    print(f"Reached max_rounds={max_rounds} without a pass, returning last result.")
    return current_result

In [3]:
DB_PATH = "DB/oedb_baseline_v3.db"
schema_path = "data/metadata_DB/schema_v3.yaml"
prompt_path_refiner_1recordT = "data/prompt_templates/refiner/prompt_refiner_1recordT.yaml"
prompt_path_ParReducer = "data/prompt_templates/prompt_ParReducer.yaml"
service_account_file = "config/service_account_key.json"
pipeline_type = "refiner_test"
prompt_path_refiner_pqa = "data/prompt_templates/refiner/prompt_refiner_pqa.yaml"

In [4]:
result = refine_notegroup(notegroup_id=1, task="answers")

Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1


2026-07-10 15:43:45 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 6465 (cached: 6400), out: 731, cost: $0.008191
2026-07-10 15:43:45 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 1) ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | F.A |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | A.A |
| Yasmine Ahmad | Female | Syria | B1 | 40 | Helmond | Y.A |
| Layla Hamliko | Female | Syria | B1 | 50 | Gemert | L.H |
| Ahmad Noman | Male | Yemen | B1 | 24 | Helmond | A.N |
| Zaid Kurami | Male | Yemen | B1 | 25 | Hemlond | Z.K |
[/TABLE]



Initial DB result — task=answers, notegroupID=1
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              answer_content_oriLAN answer_content_EN
0          1           1              3                                                                                                   

In [5]:
result = refine_notegroup(notegroup_id=2, task="answers")

Loading: Iyad - Note-taking 3.12 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Iyad - Note-taking 3.12
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1


2026-07-10 15:53:57 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 4379 (cached: 0), out: 1191, cost: $0.017384
2026-07-10 15:53:57 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 2) ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Khetam | Female | Syria | Z route | 43 | Helmond | Kh |
| Eyas | Female | Syria | B1 | 30 | Gemert | Eyas |
| Ahmad Brimo | Male | Syria | Z route, | 51 | Gemert | Ahmad |
| Nedal | Male | Syria | Z route, | 53 | Helmond | N |
| Hassan | Male | Syria | Z route, | 30 | Helmond | H |
| Wasim | Male | Syria | Zroute, | 53 | Helmond | Wasem |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | Feras |
[/TABLE]



Initial DB result — task=answers, notegroupID=2
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          answer_content_oriLAN answer_content_EN
0         91          20              7        There is a gap in communication between the school and the municipality. We tried several times to submit a complaint about the school, but the municipality did not respond. Once, I suspected that the school translated a problem and a complaint about itself to the municipality, and I later discovered t

In [6]:
result = refine_notegroup(notegroup_id=3, task="answers")

Loading: Note form Danna 22 Nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Note form Danna 22 Nov
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling


2026-07-10 15:57:25 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 6609 (cached: 0), out: 851, cost: $0.016771
2026-07-10 15:57:25 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 3) ===
[Data source: Deelnemers lijst + indeling.docx]
[TABLE]
| Name | Group | Present? | session_identifier |
| Mortada Abu Hassan | Danna - Group Arabic 1 |  | null |
| Ahmad Alhussein Alsatouf | Danna - Group Arabic 1 |  | null |
| Alaa Abdal Wahab | Danna - Group Arabic 1 |  | null |
| Lydia | Danna - Group Arabic 1 |  | L |
| Basel almoudrres | Danna - Group Arabic 1 |  | null |
[/TABLE]



Initial DB result — task=answers, notegroupID=3
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     answer_content_oriLAN answer_content_EN
0        160       

In [7]:
result = refine_notegroup(notegroup_id=4, task="answers")

Loading: Fatih notes form 22 nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Fatih notes form 22 nov
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling


2026-07-10 16:04:06 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 15563 (cached: 0), out: 936, cost: $0.028814
2026-07-10 16:04:06 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 4) ===
[Data source: Deelnemers lijst + indeling.docx]
[TABLE]
| Name | Group | Present? | session_identifier |
| Serdar Yaşar | Fatih - Groep Turks |  | S.Y |
| Ugur Yesilyurt | Fatih - Groep Turks |  | U.Y |
| Halil kalemli | Fatih - Groep Turks |  | H.K |
| M. Enes KUYUMCU | Fatih - Groep Turks |  | E.K |
| Özcan ikiz | Fatih - Groep Turks |  | O.E |
| Mehmet | Fatih - Groep Turks |  | M |
[/TABLE]



Initial DB result — task=answers, notegroupID=4
     answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [8]:
result = refine_notegroup(notegroup_id=5, task="answers")

Loading: Naya notes (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Naya notes

Initial DB result — task=answers, notegroupID=5
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [9]:
result = refine_notegroup(notegroup_id=6, task="answers")

Loading: Copy of Ale_ Note-taking form 28.11 (English translation) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Copy of Ale_ Note-taking form 28.11 (English translation)
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-10 16:32:26 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 7029 (cached: 0), out: 598, cost: $0.014766
2026-07-10 16:32:26 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 6) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 19 | Francielis Rivas | Ale - Groep spaans |  | Francielis |
[/TABLE]



Initial DB result — task=answers, notegroupID=6
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [10]:
result = refine_notegroup(notegroup_id=7, task="answers")

Loading: Danna: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Danna: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-10 16:34:36 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 7924 (cached: 0), out: 1032, cost: $0.020225
2026-07-10 16:34:36 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 7) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 11 | Rawa Alshumry | Nesrine - Group Arabic 1 |  | R |
| 12 | Abdulaziz Al-Raimi | Danna - Group Arabic 2 |  | A |
| 13 | Merry | Danna - Group Arabic 2 |  | null |
| 14 | Abdullah Najjar | Danna - Group Arabic 2 |  | A.N |
| 16 | Victor | Danna - Group Arabic 2 |  | V |
[/TABLE]



Initial DB result — task=answers, notegroupID=7
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [11]:
result = refine_notegroup(notegroup_id=8, task="answers")

Loading: Fatih: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Fatih: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-10 16:36:42 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 14263 (cached: 0), out: 684, cost: $0.024669
2026-07-10 16:36:42 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 8) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 1 | Nuri Berber | Fatih - Groep Turks |  | Nuri |
| 2 | Kamile özbek | Fatih - Groep Turks |  | Kamile |
| 3 | Kemal OZDEN | Fatih - Groep Turks |  | Kemal |
| 4 | Fatih Dogandemir | Fatih - Groep Turks |  | Fatih |
[/TABLE]



Initial DB result — task=answers, notegroupID=8
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [12]:
result = refine_notegroup(notegroup_id=9, task="answers")

Loading: Nesrine: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Nesrine: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-10 16:45:42 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 8866 (cached: 0), out: 994, cost: $0.021022
2026-07-10 16:45:42 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 9) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 7 | Ali Banat | Nesrine - Group Arabic 1 |  | Al |
| 8 | Abdelkarim Alahmad | Nesrine - Group Arabic 1 |  | Ab |
| 9 | Waleed omar Bin mahram | nesrine - Group Arabic 1 |  | W |
| 10 | Latifa Al Ajeel | Nesrine - Group Arabic 1 |  | L |
[/TABLE]



Initial DB result — task=answers, notegroupID=9
     answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [13]:
result = refine_notegroup(notegroup_id=10, task="answers")

Loading: Reza: Note-taking form 29.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Reza: Note-taking form 29.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-10 16:49:29 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 9061 (cached: 0), out: 756, cost: $0.018886
2026-07-10 16:49:29 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 10) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
| 17 | Arsalan Azarmi | Reza- Group Farsi |  | session_identifier=Arsalan |
| 18 | Aida | Reza- Group Farsi |  | session_identifier=Aida |
[/TABLE]



Initial DB result — task=answers, notegroupID=10
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [14]:
result = refine_notegroup(notegroup_id=11, task="answers")

Loading: Turkse_groep_verzamelde_data (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Turkse_groep_verzamelde_data

Initial DB result — task=answers, notegroupID=11
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [15]:
result = refine_notegroup(notegroup_id=12, task="answers")

Loading: Data session 3 (AMV, Josja) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data session 3 (AMV, Josja)

Initial DB result — task=answers, notegroupID=12
    answerID  questionID  participantID                                                                                                                                                                                                  answer_content_oriLAN answer_content_EN
0        773         270           57.0                                                                            I have to wait very long for my status, if I get one. They tell me that they will update me in three months, but now I’m nine months ahead.                  
1        774         270           57.0                                                                                                            

In [16]:
result = refine_notegroup(notegroup_id=13, task="answers")

Loading: Data den Helder Ist session Ula (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data den Helder Ist session Ula

Initial DB result — task=answers, notegroupID=13
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          answer_content_oriLAN answer_content_EN
0        831         283             58                                                                                             

In [17]:
result = refine_notegroup(notegroup_id=14, task="answers")

Loading: Zorgcafe#1_Venlo_notes.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Zorgcafe#1_Venlo_notes.docx

Initial DB result — task=answers, notegroupID=14
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           answer_content_oriLAN answer_content_EN
0        847         291           63.

In [18]:
result = refine_notegroup(notegroup_id=15, task="answers")

Loading: Notes Pepijn sessie 2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Notes Pepijn sessie 2

Initial DB result — task=answers, notegroupID=15
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 answer_content_oriLAN  

In [19]:
result = refine_notegroup(notegroup_id=16, task="answers")

Loading: Copy of iyad zorg cafe sessie 1.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Copy of iyad zorg cafe sessie 1.docx

Initial DB result — task=answers, notegroupID=16
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                               answer_content_oriLAN answer_content_EN
0        897         308             73                                                                                        My personal evaluation of them was very good. The visit was very special. Their attention to me was wonderful. They always commun

In [20]:
result = refine_notegroup(notegroup_id=17, task="answers")

Loading: Interview 3.18 BOOST (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.18 BOOST

Initial DB result — task=answers, notegroupID=17
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                            answer_content_oriLAN answer_content_EN
0        923         314             79                                                                                             Generally it’s good. Professional way they are helpful. If I have a question, I always get the answers. I don’t think they need to change anything. They close the door, it’s quiet so you concentrate. That’s good.                  
1        924         

In [21]:
result = refine_notegroup(notegroup_id=18, task="answers")

Loading: Interview 3.21 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.21

Initial DB result — task=answers, notegroupID=18
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               answer_content_oriLAN answer_content_EN
0        934         325             80                  

In [22]:
result = refine_notegroup(notegroup_id=19, task="answers")

Loading: Pepijn notulen (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/05 Expertpool/Sessie 1/Pepijn notulen

Initial DB result — task=answers, notegroupID=19
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [23]:
result = refine_notegroup(notegroup_id=20, task="answers")

Loading: NOTITIES_IYAD (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_IYAD
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-10 17:07:39 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 14059 (cached: 0), out: 2308, cost: $0.040654
2026-07-10 17:07:39 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 20) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=answers, notegroupID=20
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        answer_content_oriLAN answer_content_EN
0        969         344             89                                                                                        

In [24]:
result = refine_notegroup(notegroup_id=21, task="answers")

Loading: NOTITIES_Floris_en_Anne.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_Floris_en_Anne.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-10 17:08:32 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 22186 (cached: 0), out: 1768, cost: $0.045413
2026-07-10 17:08:32 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 21) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=answers, notegroupID=21
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [25]:
result = refine_notegroup(notegroup_id=22, task="answers")

Loading: Notites_Mahad_2.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/Notites_Mahad_2.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-10 17:09:52 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 9696 (cached: 0), out: 1773, cost: $0.029850
2026-07-10 17:09:52 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 22) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتيت


Initial DB result — task=answers, notegroupID=22
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [26]:
result = refine_notegroup(notegroup_id=23, task="answers")

Loading:  NOTITIES_Fatih (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/ NOTITIES_Fatih
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-10 17:10:39 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 15769 (cached: 0), out: 1466, cost: $0.034371
2026-07-10 17:10:39 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 23) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=answers, notegroupID=23
    answerID  questionID  participantID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [4]:
result = refine_notegroup(notegroup_id=1, task="questions")

Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1


2026-07-09 16:39:07 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 6465 (cached: 6400), out: 1084, cost: $0.011721
2026-07-09 16:39:07 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 1) ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | F.A |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | A.A |
| Yasmine Ahmad | Female | Syria | B1 | 40 | Helmond | Y.A |
| Layla Hamliko | Female | Syria | B1 | 50 | Gemert | L.H |
| Ahmad Noman | Male | Yemen | B1 | 24 | Helmond | A.N |
| Zaid Kurami | Male | Yemen | B1 | 25 | Hemlond | Z.K |
[/TABLE]



Initial DB result — task=questions, notegroupID=1
    questionID                                                                                                                                                                                                         question_content                                                                        main_indicator  followed_questionID following_trigger
0            1                                                                                                                                                                                    How is your inburgering going so far?                                                [education, onderwijs, language, taal]                  NaN               NaN
1            2                                                                                                                           Where do you feel pressure in your life, with the inburgering? What feels difficult to handle?    

In [5]:
result = refine_notegroup(notegroup_id=2, task="questions")

Loading: Iyad - Note-taking 3.12 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Iyad - Note-taking 3.12
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1


2026-07-09 16:41:06 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 4379 (cached: 0), out: 1190, cost: $0.017374
2026-07-09 16:41:06 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 2) ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Khetam | Female | Syria | Z route | 43 | Helmond | Kh |
| Eyas | Female | Syria | B1 | 30 | Gemert | Eyas |
| Ahmad Brimo | Male | Syria | Z route, | 51 | Gemert | Ahmad |
| Nedal | Male | Syria | Z route, | 53 | Helmond | N |
| Hassan | Male | Syria | Z route, | 30 | Helmond | H |
| Wasim | Male | Syria | Zroute, | 53 | Helmond | Wasem |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | Feras |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | null |
[/TABLE]



Initial DB result — task=questions, notegroupID=2
    questionID                                                                                                                      question_content             main_indicator followed_questionID following_trigger
0           20  How is your inburgering going so far? Where do you feel pressure in your life, with the inburgering? What feels difficult to handle?     [education, onderwijs]                None              None
1           21                                                            Do you feel like you want to be working at this point in your inburgering?     [work, werk & inkomen]                None              None
2           22                                               Is there space in your life to combine work/volunteer work with other responsibilities?     [work, werk & inkomen]                None              None
3           23                                                                  Does your wor

In [6]:
result = refine_notegroup(notegroup_id=3, task="questions")

Loading: Note form Danna 22 Nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Note form Danna 22 Nov
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling


2026-07-09 16:43:40 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 6609 (cached: 0), out: 940, cost: $0.017661
2026-07-09 16:43:40 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 3) ===
[Data source: Deelnemers lijst + indeling.docx]
[TABLE]
| Name | Group | Present? | session_identifier |
| Mortada Abu Hassan | Danna - Group Arabic 1 |  | null |
| Ahmad Alhussein Alsatouf | Danna - Group Arabic 1 |  | null |
| Alaa Abdal Wahab | Danna - Group Arabic 1 |  | null |
| Lydia | Danna - Group Arabic 1 |  | L |
| Basel almoudrres | Danna - Group Arabic 1 |  | null |
[/TABLE]



Initial DB result — task=questions, notegroupID=3
    questionID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           question_content          main_indicator  followed_questionID                                                    following_trigger
0           31                                                                                                                           

In [7]:
result = refine_notegroup(notegroup_id=4, task="questions")

Loading: Fatih notes form 22 nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Fatih notes form 22 nov
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling


2026-07-09 16:47:04 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 15563 (cached: 0), out: 933, cost: $0.028784
2026-07-09 16:47:04 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 4) ===
[Data source: Deelnemers lijst + indeling.docx]
[TABLE]
| Name | Group | Present? | session_identifier |
| Serdar Yaşar | Fatih - Groep Turks |  | S.Y |
| Ugur Yesilyurt | Fatih - Groep Turks |  | U.Y |
| Halil kalemli | Fatih - Groep Turks |  | H.K |
| M. Enes KUYUMCU | Fatih - Groep Turks |  | E.K |
| Özcan ikiz | Fatih - Groep Turks |  | null |
| Mehmet | Fatih - Groep Turks |  | M |
[/TABLE]



Initial DB result — task=questions, notegroupID=4
    questionID                                                                                                                                                                                                                                                                                                     question_content main_indicator  followed_questionID                                       following_trigger
0           52                                                                                                                                                                                                             If you previously had a job but are not working now:\n● Where did you work? How long did you work there?           None                  NaN                                                     NaN
1           53                                                                                                       

In [8]:
result = refine_notegroup(notegroup_id=5, task="questions")

Loading: Naya notes (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Naya notes

Initial DB result — task=questions, notegroupID=5
    questionID                                                                                                                                                                                                               question_content                                main_indicator  followed_questionID following_trigger
0           79                                                                                                                       If you previously had a job but are not working now:\n● Where did you work? How long did you work there?                                          None                  NaN               NaN
1           80                                                                                                   If y

In [9]:
result = refine_notegroup(notegroup_id=6, task="questions")

Loading: Copy of Ale_ Note-taking form 28.11 (English translation) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Copy of Ale_ Note-taking form 28.11 (English translation)
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 16:49:36 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 7029 (cached: 0), out: 789, cost: $0.016676
2026-07-09 16:49:36 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 6) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 19 | Francielis Rivas | Ale - Groep spaans |  | Francielis |
[/TABLE]



Initial DB result — task=questions, notegroupID=6
    questionID                                                                                                                                                                                                  question_content   main_indicator  followed_questionID                           following_trigger
0          108                                                                                                                                                                                  Where do you work at the moment?             None                  NaN                                         NaN
1          109                                                                                                                       How did you find this job (through which method)? Why did you choose that way of searching?             None                108.0             Participant currently has a job
2          110              

In [10]:
result = refine_notegroup(notegroup_id=7, task="questions")

Loading: Danna: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Danna: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 16:51:14 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 7924 (cached: 0), out: 886, cost: $0.018765
2026-07-09 16:51:14 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 7) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 11 | Rawa Alshumry | Nesrine - Group Arabic 1 |  | R |
| 12 | Abdulaziz Al-Raimi | Danna - Group Arabic 2 |  | A |
| 13 | Merry | Danna - Group Arabic 2 |  | Merry |
| 14 | Abdullah Najjar | Danna - Group Arabic 2 |  | A.N |
| 16 | Victor | Danna - Group Arabic 2 |  | V |
[/TABLE]



Initial DB result — task=questions, notegroupID=7
    questionID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [11]:
result = refine_notegroup(notegroup_id=8, task="questions")

Loading: Fatih: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Fatih: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 16:55:30 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 14263 (cached: 0), out: 892, cost: $0.026749
2026-07-09 16:55:30 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 8) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 1 | Nuri Berber | Fatih - Groep Turks |  | Nuri |
| 2 | Kamile özbek | Fatih - Groep Turks |  | Kamile |
| 3 | Kemal OZDEN | Fatih - Groep Turks |  | Kemal |
| 4 | Fatih Dogandemir | Fatih - Groep Turks |  | Fatih |
[/TABLE]



Initial DB result — task=questions, notegroupID=8
    questionID                                                                                                                                                                                                                                                                                                         question_content          main_indicator  followed_questionID                                         following_trigger
0          157                                                                                                                                                                                                                    If you previously had a job but are not working now: Where did you work? How long did you work there?                    None                156.0  People who have had a job before but are not working now
1          158                                                                         

In [12]:
result = refine_notegroup(notegroup_id=9, task="questions")

Loading: Nesrine: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Nesrine: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 16:58:31 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 8866 (cached: 0), out: 714, cost: $0.018222
2026-07-09 16:58:31 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 9) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
| 7 | Ali Banat | Nesrine - Group Arabic 1 |  | session_identifier=Al |
| 8 | Abdelkarim Alahmad | Nesrine - Group Arabic 1 |  | session_identifier=Ab |
| 9 | Waleed omar Bin mahram | nesrine - Group Arabic 1 |  | session_identifier=W |
| 10 | Latifa Al Ajeel | Nesrine - Group Arabic 1 |  | session_identifier=L |
[/TABLE]



Initial DB result — task=questions, notegroupID=9
    questionID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [13]:
result = refine_notegroup(notegroup_id=10, task="questions")

Loading: Reza: Note-taking form 29.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Reza: Note-taking form 29.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 17:03:01 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 9061 (cached: 0), out: 580, cost: $0.017126
2026-07-09 17:03:01 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 10) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 17 | Arsalan Azarmi | Reza- Group Farsi |  | Arsalan |
| 18 | Aida | Reza- Group Farsi |  | Aida |
[/TABLE]



Initial DB result — task=questions, notegroupID=10
    questionID                                                                                                                                                                                                                                                                                                            question_content          main_indicator  followed_questionID                                                                     following_trigger
0          222                                                                                                                                                                                                                      If you previously had a job but are not working now:\nWhere did you work? How long did you work there?                    None                  NaN                                                                                   NaN
1          223          

In [14]:
result = refine_notegroup(notegroup_id=11, task="questions")

Loading: Turkse_groep_verzamelde_data (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Turkse_groep_verzamelde_data

Initial DB result — task=questions, notegroupID=11
    questionID                                                                                 question_content main_indicator followed_questionID following_trigger
0          254                          Choose 3 main indicators regarding your biggest concerns at the moment.           None                None              None
1          255  Waarom heb je deze indicator gekozen? Waarom is deze indicator belangrijk voor jou persoonlijk?           None                None              None
2          256        Wat wil je bereiken op deze indicator? Wat is je gewenste toekomst m.b.t. deze indicator?           None                None              None
3          257               

In [15]:
result = refine_notegroup(notegroup_id=12, task="questions")

Loading: Data session 3 (AMV, Josja) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data session 3 (AMV, Josja)

Initial DB result — task=questions, notegroupID=12
    questionID                                                                                            question_content main_indicator followed_questionID following_trigger
0          270                                                                                                   Stability    [stability]                None              None
1          271                                                                                                     Leisure      [leisure]                None              None
2          272                                                                                                      Health       [health]                None      

In [16]:
result = refine_notegroup(notegroup_id=13, task="questions")

Loading: Data den Helder Ist session Ula (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data den Helder Ist session Ula

Initial DB result — task=questions, notegroupID=13
   questionID                                                                                                                     question_content                   main_indicator followed_questionID following_trigger
0         283              Languages\nWhy did you choose this indicator?\nAchieve?\nObstacles?\nActions taken?\nWhat would help achieve this goal?                 [language, taal]                None              None
1         284                Housing\nWhy did you choose this indicator?\nAchieve?\nObstacles?\nActions taken?\nWhat would help achieve this goal?  [housing, huisvesting & opvang]                None              None
2         285                

In [17]:
result = refine_notegroup(notegroup_id=14, task="questions")

Loading: Zorgcafe#1_Venlo_notes.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Zorgcafe#1_Venlo_notes.docx

Initial DB result — task=questions, notegroupID=14
   questionID                                                                                                                               question_content main_indicator followed_questionID following_trigger
0         291  What are the good things that are happening? Things that can get better? Experience as a volunteer. What is important for you as a volunteer?           None                None              None
1         292                                                                                      What are the repetitive people and problems that you see?           None                None              None
2         293                                           Knowledge over t

In [18]:
result = refine_notegroup(notegroup_id=15, task="questions")

Loading: Notes Pepijn sessie 2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Notes Pepijn sessie 2

Initial DB result — task=questions, notegroupID=15
   questionID                                                                                                           question_content main_indicator followed_questionID following_trigger
0         297  How was it to keep the notebook and write notes. Who did this? For those who did: did it help to be more aware of health?           None                None              None
1         298                                               How are you feeling health-wise? Are you feeling better since Zorgcafé? Why?           None                None              None
2         300     Do you have the feeling that improved health has an effect on ability to care for yourself/make independent decisions?           None                None        

In [19]:
result = refine_notegroup(notegroup_id=16, task="questions")

Loading: Copy of iyad zorg cafe sessie 1.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Copy of iyad zorg cafe sessie 1.docx

Initial DB result — task=questions, notegroupID=16
   questionID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [20]:
result = refine_notegroup(notegroup_id=17, task="questions")

Loading: Interview 3.18 BOOST (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.18 BOOST

Initial DB result — task=questions, notegroupID=17
    questionID                                                                                                                                                                                                  question_content main_indicator followed_questionID following_trigger
0          314                                                                                                                                   Wat vond je van het taalcafé van BOOST? Wat denk je dat er beter kan en waarom?           None                None              None
1          315                                                                                                                                                    Over welke thema’s heb je h

In [21]:
result = refine_notegroup(notegroup_id=18, task="questions")

Loading: Interview 3.21 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.21

Initial DB result — task=questions, notegroupID=18
    questionID                                                                                                                                                                                                  question_content main_indicator followed_questionID following_trigger
0          325                                                                                                                                   Wat vond je van het taalcafé van BOOST? Wat denk je dat er beter kan en waarom?           None                None              None
1          326                                                                                                                                                    Over welke thema’s heb je het meeste ge

In [22]:
result = refine_notegroup(notegroup_id=19, task="questions")

Loading: Pepijn notulen (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/05 Expertpool/Sessie 1/Pepijn notulen

Initial DB result — task=questions, notegroupID=19
   questionID                                                                                                                                                           question_content main_indicator followed_questionID following_trigger
0         336                                                                                                                                               Rating inburgeringsprogramma           None                None              None
1         337                                                                                                                                                          STOP/START/REPEAT           None                None              None
2         340                                             

In [23]:
result = refine_notegroup(notegroup_id=20, task="questions")

Loading: NOTITIES_IYAD (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_IYAD
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-10 11:35:44 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 14059 (cached: 0), out: 2952, cost: $0.047094
2026-07-10 11:35:44 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 20) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=questions, notegroupID=20
    questionID                                                                                                                                                                                                                                                                                                                                                                               question_content main_indicator followed_questionID following_trigger
0          344                                                                                                                                                                                                                                                    How do you experience the integration (inburgering) process as a whole (language, MAP, guidance, activities)? What helps you make progress?           None                None              None
1          345                                

In [24]:
result = refine_notegroup(notegroup_id=21, task="questions")

Loading: NOTITIES_Floris_en_Anne.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_Floris_en_Anne.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-10 11:37:55 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 22186 (cached: 0), out: 1654, cost: $0.044272
2026-07-10 11:37:55 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 21) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=questions, notegroupID=21
    questionID                                                                                                                                                                                                                                                                                                                                                                               question_content                                               main_indicator followed_questionID following_trigger
0          364                                                                                                                                                                                                                                                                                       Kun je jezelf kort voorstellen? (Naam, land van herkomst,\nhoelang in Nederland, gezin of alleenstaand?)                                                      

In [25]:
result = refine_notegroup(notegroup_id=22, task="questions")

Loading: Notites_Mahad_2.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/Notites_Mahad_2.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-10 11:57:54 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 9696 (cached: 0), out: 1968, cost: $0.031800
2026-07-10 11:57:54 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 22) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتيت


Initial DB result — task=questions, notegroupID=22
    questionID                                                                                                                                                                         question_content main_indicator followed_questionID following_trigger
0          387                                                                                                                                                   Wat is je favoriete Nederlandse woord?           None                None              None
1          388                                                                                  Kun je jezelf kort voorstellen?(Naam, land van herkomst, hoe lang in Nederland, gezin of alleenstaand?)           None                None              None
2          389                                                          Hoe ervaar je het inburgeringsproces als geheel (taal, MAP, begeleiding, activiteiten)? Wat helpt jou om vooruitgang 

In [26]:
result = refine_notegroup(notegroup_id=23, task="questions")

Loading:  NOTITIES_Fatih (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/ NOTITIES_Fatih
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-10 12:01:51 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 15769 (cached: 0), out: 1260, cost: $0.032311
2026-07-10 12:01:51 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 23) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=questions, notegroupID=23
    questionID                                                                                                                                                                                                                                                                                                                                                                                           question_content                                               main_indicator followed_questionID following_trigger
0          420                                                                                                                                                                                                                                                                            Hoe ervaar je het inburgeringsproces als geheel (taal, MAP, begeleiding, activiteiten)? Wat helpt jou om vooruitgang te boeken?           [taal, onderwijs, r

In [28]:
result = refine_notegroup(notegroup_id=1, task="participants")

Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1


2026-07-09 15:32:53 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 6465 (cached: 0), out: 1068, cost: $0.018761
2026-07-09 15:32:53 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 1) ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | F.A |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | A.A |
| Yasmine Ahmad | Female | Syria | B1 | 40 | Helmond | Y.A |
| Layla Hamliko | Female | Syria | B1 | 50 | Gemert | L.H |
| Ahmad Noman | Male | Yemen | B1 | 24 | Helmond | A.N |
| Zaid Kurami | Male | Yemen | B1 | 25 | Hemlond | Z.K |
[/TABLE]



Initial DB result — task=participants, notegroupID=1
   participantID      full_name session_identifier  gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0              1    Ahmad Ahmad                A.A    Male  35.0             B1              None   None           Syria           None               None      Helmond                {}
1              2  Yasmine Ahmad                Y.A  Female  40.0             B1              None   None           Syria           None               None      Helmond                {}
2              3  Layla Hamliko                L.H  Female  50.0             B1              None   None           Syria           None               None       Gemert                {}
3              4    Ahmad Noman                A.N    Male  24.0             B1              None   None           Yemen           None               None      Helmond                {}
4              5

In [29]:
result = refine_notegroup(notegroup_id=2, task="participants")

Loading: Iyad - Note-taking 3.12 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Iyad - Note-taking 3.12
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1


2026-07-09 15:35:25 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 4379 (cached: 0), out: 1287, cost: $0.018344
2026-07-09 15:35:25 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 2) ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Khetam | Female | Syria | Z route | 43 | Helmond | Kh |
| Eyas | Female | Syria | B1 | 30 | Gemert | Eyas |
| Ahmad Brimo | Male | Syria | Z route, | 51 | Gemert | Ahmad |
| Nedal | Male | Syria | Z route, | 53 | Helmond | N |
| Hassan | Male | Syria | Z route, | 30 | Helmond | H |
| Wasim | Male | Syria | Zroute, | 53 | Helmond | Wasem |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | Feras |
[/TABLE]



Initial DB result — task=participants, notegroupID=2
    participantID full_name session_identifier  gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0               7    Khetam                 Kh  Female  43.0        Z route              None   None           Syria           None               None      Helmond                {}
1               8      Eyas               Eyas  Female  30.0             B1              None   None           Syria           None               None       Gemert                {}
2              10     Nedal                  N    Male  53.0       Z route,              None   None           Syria           None               None      Helmond                {}
3              11    Hassan                  H    Male  30.0       Z route,              None   None           Syria           None               None      Helmond                {}
4              17       NaN         

In [30]:
result = refine_notegroup(notegroup_id=3, task="participants")

Loading: Note form Danna 22 Nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Note form Danna 22 Nov
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling


2026-07-09 15:39:40 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 6609 (cached: 0), out: 144, cost: $0.009701
2026-07-09 15:39:40 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 3) ===
[Data source: Deelnemers lijst + indeling.docx]
[TABLE]
| Name | Group | Present? | session_identifier |
| Mortada Abu Hassan | Danna - Group Arabic 1 |  | M.A. |
| Ahmad Alhussein Alsatouf | Danna - Group Arabic 1 |  | A.A. |
| Alaa Abdal Wahab | Danna - Group Arabic 1 |  | A.A.2 |
| Lydia | Danna - Group Arabic 1 |  | L |
| Basel almoudrres | Danna - Group Arabic 1 |  | null |
[/TABLE]



Initial DB result — task=participants, notegroupID=3
   participantID                 full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             24        Mortada Abu Hassan               M.A.   None  None           None              None   None            None       [Arabic]               None        Gouda              None
1             25  Ahmad Alhussein Alsatouf               A.A.   None  None           None              None   None            None       [Arabic]               None          NaN              None
2             26          Alaa Abdal Wahab              A.A.2   None  None           None              None   None            None       [Arabic]               None      Haarlem              None
3             27                     Lydia                  L   None  None           None              None   None            None           None               No

In [31]:
result = refine_notegroup(notegroup_id=4, task="participants")

Loading: Fatih notes form 22 nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Fatih notes form 22 nov
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling


2026-07-09 15:41:41 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 15563 (cached: 0), out: 359, cost: $0.023044
2026-07-09 15:41:41 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 4) ===
[Data source: Deelnemers lijst + indeling.docx]
[TABLE]
| Name | Group | Present? | session_identifier |
| Serdar Yaşar | Fatih - Groep Turks |  | S.Y |
| Ugur Yesilyurt | Fatih - Groep Turks |  | U.Y |
| Halil kalemli | Fatih - Groep Turks |  | H.K |
| M. Enes KUYUMCU | Fatih - Groep Turks |  | E.K |
| Özcan ikiz | Fatih - Groep Turks |  | O.E |
| Mehmet | Fatih - Groep Turks |  | M |
| Bekir Akgül | Fatih - Groep Turks |  | null |
| Mortada Abu Hassan | Danna - Group Arabic 1 |  | null |
| Ahmad Alhussein Alsatouf | Danna - Group Arabic 1 |  | null |
| Alaa Abdal Wahab | Danna - Group Arabic 1 |  | null |
| Lydia | Danna - Group Arabic 1 |  | null |
| Basel almoudrres | Danna - Group Arabic 1 |  | null |
| Moham


Initial DB result — task=participants, notegroupID=4
   participantID        full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             28     Serdar Yaşar                S.Y   None  None           None              None   None            None        [Turks]               None         None              None
1             29   Ugur Yesilyurt                U.Y   None  None           None              None   None            None        [Turks]               None         None              None
2             30    Halil kalemli                H.K   None  None           None              None   None            None        [Turks]               None         None              None
3             31  M. Enes KUYUMCU                E.K   None  None           None              None   None            None        [Turks]               None         None              None
4          

In [32]:
result = refine_notegroup(notegroup_id=5, task="participants")

Loading: Naya notes (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Naya notes

Initial DB result — task=participants, notegroupID=5
   participantID full_name session_identifier gender   age learning_route participant_group status place_of_origin    language_group first_arrival_date municipality other_information
0             34     Adeel              Adeel   None  None           None     Asylum seeker   None        Pakistan  [English, Dutch]               None      Zaandam              None
1             35   Arsalan            Arsalan   None  None           None     Asylum seeker   None        Pakistan  [English, Dutch]               None      Zaandam              None

Round 1/6 — task=participants, notegroupID=5
   participantID full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality oth

In [33]:
result = refine_notegroup(notegroup_id=6, task="participants")

Loading: Copy of Ale_ Note-taking form 28.11 (English translation) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Copy of Ale_ Note-taking form 28.11 (English translation)
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 15:48:17 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 7029 (cached: 0), out: 72, cost: $0.009506
2026-07-09 15:48:17 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 6) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 19 | Francielis Rivas | Ale - Groep spaans |  | Francielis |
[/TABLE]



Initial DB result — task=participants, notegroupID=6
   participantID         full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             36  Francielis Rivas         Francielis   None  None       B1-route              None   None       Venezuela       [Spaans]         2021-01-01    Amsterdam                {}

Round 1/6 — task=participants, notegroupID=6
   participantID         full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             36  Francielis Rivas         Francielis   None  None       B1-route              None   None       Venezuela       [Spaans]               None    Amsterdam                {}

Round 2/6 — task=participants, notegroupID=6
   participantID         full_name session_identifier gender   age learning_route participant_group st

In [34]:
result = refine_notegroup(notegroup_id=7, task="participants")

Loading: Danna: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Danna: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 15:50:39 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 7924 (cached: 0), out: 155, cost: $0.011455
2026-07-09 15:50:39 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 7) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 11 | Rawa Alshumry | Nesrine - Group Arabic 1 |  | R |
| 12 | Abdulaziz Al-Raimi | Danna - Group Arabic 2 |  | Abdulaziz |
| 13 | Merry | Danna - Group Arabic 2 |  | Merry |
| 14 | Abdullah Najjar | Danna - Group Arabic 2 |  | A.N |
| 16 | Victor | Danna - Group Arabic 2 |  | V |
[/TABLE]



Initial DB result — task=participants, notegroupID=7
   participantID           full_name session_identifier gender   age learning_route participant_group status place_of_origin              language_group first_arrival_date municipality other_information
0             37       Rawa Alshumry                  R   None   NaN           None              None   None             NaN                    [Arabic]                NaN    Rotterdam                {}
1             38  Abdulaziz Al-Raimi                NaN   None   NaN           None              None   None             NaN                    [Arabic]                NaN    Groningen                {}
2             39     Abdullah Najjar                A.N   None   NaN           None              None   None           Syria  [Arabic, English, Turkish]                NaN      Utrecht                {}
3             40              Victor                  V   None  41.0           None              None   None           Syria          

In [35]:
result = refine_notegroup(notegroup_id=8, task="participants")

Loading: Fatih: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Fatih: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 15:54:02 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 14263 (cached: 0), out: 690, cost: $0.024729
2026-07-09 15:54:02 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 8) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 1 | Nuri Berber | Fatih - Groep Turks |  | Nuri |
| 2 | Kamile özbek | Fatih - Groep Turks |  | Kamile |
| 3 | Kemal OZDEN | Fatih - Groep Turks |  | Kemal |
| 4 | Fatih Dogandemir | Fatih - Groep Turks |  | Fatih |
[/TABLE]



Initial DB result — task=participants, notegroupID=8
   participantID         full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             41       Nuri Berber               Nuri   None   NaN           None               NaN   None             NaN        [Turks]               None       Raalte              None
1             42      Kamile özbek             Kamile   None   NaN           None               NaN   None         Turkije        [Turks]               None          NaN              None
2             43       Kemal OZDEN              Kemal   None  50.0           None               NaN   None         Turkije        [Turks]               None       Raalte              None
3             44  Fatih Dogandemir              Fatih   None   NaN           None     Permit holder   None         Turkije        [Turks]               None          NaN              None

Round

In [36]:
result = refine_notegroup(notegroup_id=9, task="participants")

Loading: Nesrine: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Nesrine: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 15:58:44 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 8866 (cached: 0), out: 869, cost: $0.019772
2026-07-09 15:58:44 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 9) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 7 | Ali Banat | Nesrine - Group Arabic 1 |  | Al |
| 8 | Abdelkarim Alahmad | Nesrine - Group Arabic 1 |  | Ab |
| 9 | Waleed omar Bin mahram | nesrine - Group Arabic 1 |  | W |
| 10 | Latifa Al Ajeel | Nesrine - Group Arabic 1 |  | L |
[/TABLE]



Initial DB result — task=participants, notegroupID=9
   participantID               full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             45               Ali Banat                 Al   None  None            NaN              None   None           Syria       [Arabic]               None     Den Haag              None
1             46      Abdelkarim Alahmad                 Ab   None  None       B1-route              None   None           Syria       [Arabic]               None          NaN              None
2             47  Waleed omar Bin mahram                  W   None  None            NaN              None   None             NaN       [Arabic]               None        Venlo              None
3             48         Latifa Al Ajeel                  L   None  None            NaN              None   None             NaN       [Arabic]               None    Gron

In [38]:
result = refine_notegroup(notegroup_id=10, task="participants")

Loading: Reza: Note-taking form 29.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Reza: Note-taking form 29.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2


2026-07-09 16:02:37 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 9061 (cached: 0), out: 440, cost: $0.015726
2026-07-09 16:02:37 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 10) ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 17 | Arsalan Azarmi | Reza- Group Farsi |  | Arsalan |
| 18 | Aida | Reza- Group Farsi |  | Aida |
[/TABLE]



Initial DB result — task=participants, notegroupID=10
   participantID       full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             49  Arsalan Azarmi            Arsalan   None  None           None              None   None            None        [Farsi]               None    Rotterdam              None
1             50            Aida               Aida   None  None           None              None   None            None        [Farsi]               None          NaN              None

Round 1/6 — task=participants, notegroupID=10
   participantID       full_name session_identifier gender   age learning_route participant_group status place_of_origin   language_group first_arrival_date municipality other_information
0             49  Arsalan Azarmi            Arsalan   None  None           None              None   None            None          [Farsi]               

In [39]:
result = refine_notegroup(notegroup_id=11, task="participants")

Loading: Turkse_groep_verzamelde_data (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Turkse_groep_verzamelde_data

Initial DB result — task=participants, notegroupID=11
   participantID full_name             session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             51      None                              E  Vrouw  None           None       Asielzoeker   None           Turks        [Turks]               None   Den Helder              None
1             52      None                              A    Man  None           None       Asielzoeker   None           Turks        [Turks]               None   Den Helder              None
2             53      None  S, Turkse, Asielzoeker, Vrouw  Vrouw  None           None       Asielzoeker   None

In [14]:
result = refine_notegroup(notegroup_id=12, task="participants")

Loading: Data session 3 (AMV, Josja) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data session 3 (AMV, Josja)

Initial DB result — task=participants, notegroupID=12
   participantID full_name session_identifier gender  age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             55      None      Participant 1  woman   19           None              None   None            Iraq           None               None   Den Helder                {}
1             56      None      Participant 2  woman   17           None              None   None         Somalia           None               None   Den Helder                {}
2             57      None      Participant 3    man   19           None              None   None        Sudanese           None               None   D

In [15]:
result = refine_notegroup(notegroup_id=13, task="participants")

Loading: Data den Helder Ist session Ula (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data den Helder Ist session Ula

Initial DB result — task=participants, notegroupID=13
   participantID full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             58     Afram               None   None  None           None              None   None           Syria           None               None   Den Helder              None
1             59     Jalal               None   None  None           None              None   None             NaN           None               None   Den Helder              None
2             60    Khaled               None   None  None           None              None   None             NaN           None           

In [16]:
result = refine_notegroup(notegroup_id=14, task="participants")

Loading: Zorgcafe#1_Venlo_notes.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Zorgcafe#1_Venlo_notes.docx

Initial DB result — task=participants, notegroupID=14
   participantID full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             62      None               S(1)    man  None           None              None   None           Iraqi           None               None        Venlo              None
1             63      None               F(2)  woman  None           None              None   None             NaN        [Farsi]               None        Venlo              None
2             64      None               T(3)    man  None           None              None   None         Turkish      [Turkish]               None        Ven

In [41]:
result = refine_notegroup(notegroup_id=15, task="participants")

Loading: Notes Pepijn sessie 2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Notes Pepijn sessie 2

Initial DB result — task=participants, notegroupID=15
   participantID        full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             66            Ahmad        A. B1. Camp   None  None       B1-route               NaN   None             NaN           None               None        Venlo              None
1             67  Maziad Ghaibour        M. B1. Camp   None  None       B1-route               NaN   None             NaN           None               None        Venlo              None
2             68  Mahmoud Aljabri      MG. B1. House   None  None       B1-route               NaN   None             NaN           None               None        Venlo              Non

In [42]:
result = refine_notegroup(notegroup_id=16, task="participants")

Loading: Copy of iyad zorg cafe sessie 1.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Copy of iyad zorg cafe sessie 1.docx

Initial DB result — task=participants, notegroupID=16
   participantID        full_name session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             73            Ahmad        A. B1. Camp   None  None       B1-route              None   None            None           None               None        Venlo              None
1             74  Maziad Ghaibour        M. B1. Camp   None  None       B1-route              None   None            None           None               None        Venlo              None
2             75  Mahmoud Aljabri      MG. B1. House   None  None       B1-route              None   None            Non

In [19]:
result = refine_notegroup(notegroup_id=17, task="participants")

Loading: Interview 3.18 BOOST (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.18 BOOST

Initial DB result — task=participants, notegroupID=17
   participantID full_name    session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             79      None  Interview 3.18 BOOST   None  None           None              None   None            None           None               None    Amsterdam              None

Round 1/3 — task=participants, notegroupID=17
   participantID full_name    session_identifier gender   age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             79      None  Interview 3.18 BOOST   None  None           None              None   None            None           None              

In [20]:
result = refine_notegroup(notegroup_id=18, task="participants")

Loading: Interview 3.21 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.21

Initial DB result — task=participants, notegroupID=18
   participantID full_name session_identifier gender   age learning_route participant_group status place_of_origin      language_group first_arrival_date municipality other_information
0             80      None     Interview 3.21   None  None           None              None   None            None  [Swahili, English]               None    Amsterdam                {}

Round 1/3 — task=participants, notegroupID=18
   participantID full_name session_identifier gender   age learning_route participant_group status place_of_origin      language_group first_arrival_date municipality other_information
0             80      None     Interview 3.21   None  None           None              None   None            None  [Swahili, English]               Non

In [21]:
result = refine_notegroup(notegroup_id=19, task="participants")

Loading: Pepijn notulen (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/05 Expertpool/Sessie 1/Pepijn notulen

Initial DB result — task=participants, notegroupID=19
   participantID full_name session_identifier gender  age learning_route participant_group status place_of_origin language_group first_arrival_date municipality other_information
0             81      None                 M.    Man   28       B1-route              None   None           Syrië     [Arabisch]         2022-01-01    Amsterdam                {}
1             82      None                 A.    Man   38       B1-route              None   None          Egypte       [Engels]         2018-01-01    Amsterdam                {}
2             83      None                 H.    Man   43       Oude wet              None   None      Bangladesh       [Engels]         2019-01-01    Amsterdam                {}
3             84      None                 N.  Vrouw 

In [22]:
result = refine_notegroup(notegroup_id=20, task="participants")

Loading: NOTITIES_IYAD (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_IYAD
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-09 15:05:29 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 14059 (cached: 0), out: 1999, cost: $0.037564
2026-07-09 15:05:29 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 20) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=participants, notegroupID=20
   participantID         full_name session_identifier                                                                                                                                                             gender  age  learning_route participant_group status place_of_origin        language_group first_arrival_date                            municipality other_information
0             85    Amer Al-maleki                  A                                                                                                                                            Man / Man / رجل / Erkek   30        B1-route              None   None           Jemen            [Arabisch]               None                            Monnickendam                {}
1             86  ‪Reem Al abbas‬‏                  R                                                                                                                                      Vr

In [23]:
result = refine_notegroup(notegroup_id=21, task="participants")

Loading: NOTITIES_Floris_en_Anne.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_Floris_en_Anne.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-09 15:08:42 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 22186 (cached: 0), out: 1934, cost: $0.047073
2026-07-09 15:08:42 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 21) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=participants, notegroupID=21
   participantID                      full_name session_identifier                         gender  age  learning_route participant_group status place_of_origin language_group first_arrival_date  municipality other_information
0             91                Musfira Mahnoor      Vrouw, 23, B1  Vrouw / Woman / امرأة / Kadın   23  Onderwijsroute    Family migrant   None        Pakistan      [English]               None      Ilpendam                {}
1             92  Angela Paola Barragán sanchez      Vrouw, 42, B1  Vrouw / Woman / امرأة / Kadın   42        B1-route    Family migrant   None       ColombiaC      [Español]               None  Monnickendam                {}

Round 1/3 — task=participants, notegroupID=21
   participantID                      full_name session_identifier                         gender  age  learning_route participant_group status place_of_origin language_group first_arrival_date  municipality other_inform

In [24]:
result = refine_notegroup(notegroup_id=22, task="participants")

Loading: Notites_Mahad_2.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/Notites_Mahad_2.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-09 15:11:03 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 9696 (cached: 0), out: 985, cost: $0.021970
2026-07-09 15:11:03 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 22) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتيت 


Initial DB result — task=participants, notegroupID=22
   participantID             full_name session_identifier                                                                                                                                                             gender  age learning_route participant_group status place_of_origin                             language_group first_arrival_date  municipality other_information
0             93        Mohamed Hardan          Mohamed 1                                                                                                                                            Man / Man / رجل / Erkek   45        Z-route              None   None           Syrië                                 [Arabisch]               None  Monnickendam                {}
1             94           Hala Hardan              Haala                                                                                                                                      Vr

In [25]:
result = refine_notegroup(notegroup_id=23, task="participants")

Loading:  NOTITIES_Fatih (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/ NOTITIES_Fatih
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-09 15:29:26 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 15769 (cached: 0), out: 1280, cost: $0.032511
2026-07-09 15:29:26 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 23) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي


Initial DB result — task=participants, notegroupID=23
   participantID                 full_name session_identifier                         gender  age learning_route participant_group status place_of_origin language_group first_arrival_date  municipality other_information
0             98  Mustafa Aykut Alp Yılmaz              Aykut        Man / Man / رجل / Erkek   46       B1-route              None   None         Türkiye       [Türkçe]               None     Landsmeer                {}
1             99                      Elif               Elif  Vrouw / Woman / امرأة / Kadın   42       B1-route              None   None         Turkiye       [Turkce]               None  Monnickendam                {}

Round 1/3 — task=participants, notegroupID=23
   participantID                 full_name session_identifier                         gender  age learning_route participant_group status place_of_origin language_group first_arrival_date  municipality other_information
0             98  

In [6]:
result = refine_notegroup(notegroup_id=17)
print(result)

Loading: Interview 3.18 BOOST (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.18 BOOST
--- Initial DB result for notegroupID=17 ---
{"date": null, "data_source_category": "diepteinterview"}
--- Round 1/3 result for notegroupID=17 ---
{"date": "pass", "data_source_category": "pass"}
--- Passed on round 1, returning previous result ---
{"date": null, "data_source_category": "diepteinterview"}


In [7]:
result = refine_notegroup(notegroup_id=21)
print(result)

Loading: NOTITIES_Floris_en_Anne.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_Floris_en_Anne.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-02 13:03:08 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 22186 (cached: 0), out: 722, cost: $0.034952
2026-07-02 13:03:08 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 21) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتيت

--- Initial DB result for notegroupID=21 ---
{"date": null, "data_source_category": "focus group"}
--- Round 1/3 result for notegroupID=21 ---
{"date": "pass", "data_source_category": "pass"}
--- Passed on round 1, returning previous result ---
{"date": null, "data_source_category": "focus group"}


In [8]:
result = refine_notegroup(notegroup_id=12)
print(result)

Loading: Data session 3 (AMV, Josja) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data session 3 (AMV, Josja)
--- Initial DB result for notegroupID=12 ---
{"date": "2024-05-14", "data_source_category": "expertpool"}
--- Round 1/3 result for notegroupID=12 ---
{"date": "pass", "data_source_category": "pass"}
--- Passed on round 1, returning previous result ---
{"date": "2024-05-14", "data_source_category": "expertpool"}


In [9]:
result = refine_notegroup(notegroup_id=20)
print(result)

Loading: NOTITIES_IYAD (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_IYAD
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-02 13:04:24 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 14059 (cached: 0), out: 1003, cost: $0.027604
2026-07-02 13:04:24 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 20) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتي

--- Initial DB result for notegroupID=20 ---
{"date": null, "data_source_category": null}
--- Round 1/3 result for notegroupID=20 ---
{"date": null, "data_source_category": "focus group"}
--- Round 2/3 result for notegroupID=20 ---
{"date": "pass", "data_source_category": "pass"}
--- Passed on round 2, returning previous result ---
{"date": null, "data_source_category": "focus group"}


In [ ]:
from oral_notes.prompt_combiner_v3 import PromptCombiner

schema_path = "data/metadata_DB/schema.yaml"
combiner = PromptCombiner(schema_path=schema_path)

result = combiner.build_pass_placeholder("answers")
print(result)

In [ ]:
db_path = "DB/oedb_baseline_v3.db"
notegroup_id = 1

for table in ["notegroups", "participants", "questions", "answers"]:
    result = fetch_notegroup_json(db_path, table, notegroup_id)
    print(f"--- {table} ---")
    print(result)
    print()